# Session 8 — Structure Meets Function

**Goal of this session:** put the structural connectome from session 7 next to a functional connectivity network, edge by edge, and talk honestly about how strongly — and how weakly — they actually relate.

*Network Neuroscience in Python, session 8 of 10.*

## Why this matters

It's tempting to assume that if two regions are physically wired together by white matter, their activity should correlate strongly, and if they aren't wired together, it shouldn't. Real brains are messier than that. Two regions with no direct fibre tract can still show strong functional coupling through an indirect path (A→C→B), and two directly wired regions can show weak functional correlation if what they're exchanging isn't reflected in slow BOLD fluctuations. The relationship between structure and function is real, extensively studied, and nowhere near a 1-to-1 mapping. This session measures that relationship directly instead of assuming it.

## Two honesty problems, both worth stating before we start

**Different people.** The structural connectome is the Stanford HARDI subject from session 7. The functional connectivity matrix below comes from one adult participant in nilearn's open "development fMRI" dataset — a completely different, unrelated person, from a different study. A properly controlled version of this analysis needs diffusion and resting-state scans from the *same* people. We don't have that here, and we're not going to pretend otherwise: this notebook demonstrates the method, not a real within-subject finding.

**Different atlases.** Session 7's regions come from a Desikan-Killiany-style parcellation drawn directly on the Stanford subject's own anatomy. The functional data below is extracted with AAL3, a separate, standard MNI-space anatomical atlas, because no atlas ships both as a labelled volume in MNI space *and* pre-registered to the Stanford subject specifically. `scripts/build_functional_for_comparison.py` matches the two atlases' regions by anatomical *name* (e.g. "superiorfrontal" ↔ "Frontal_Sup_2"), not by shared voxels — a curated, approximate correspondence for the regions where a confident name match exists, skipping the rest. That script is short; read it if you want to see exactly which 70 of the 86 structural regions made the cut and why the other 16 didn't.

In [ ]:
import io
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO_RAW = "https://raw.githubusercontent.com/saeedrafsharx/network-neuroscience-python/main/data/"
DATA = "../data/" if os.path.exists("../data/functional_connectome_matched.npy") else REPO_RAW
print("reading cached data from:", DATA)


def load_array(name):
    if DATA.startswith("http"):
        with urllib.request.urlopen(DATA + name) as response:
            return np.load(io.BytesIO(response.read()))
    return np.load(DATA + name)


def load_text(name):
    if DATA.startswith("http"):
        with urllib.request.urlopen(DATA + name) as response:
            return response.read().decode().strip()
    with open(DATA + name) as f:
        return f.read().strip()


structural = load_array("structural_connectome_matched.npy")
functional = load_array("functional_connectome_matched.npy")
region_labels = pd.read_csv(DATA + "comparison_region_labels.csv")
functional_subject = load_text("comparison_functional_subject.txt")

print(f"structural matrix: {structural.shape}   functional matrix: {functional.shape}")
print(f"{len(region_labels)} matched regions")
print(f"functional subject: {functional_subject}  (structural subject: the Stanford HARDI subject — different person)")

## The two matrices, side by side

Same region order in both, so the same row/column means the same anatomical region in each matrix. Notice how different the two look even by eye: the structural matrix is sparse and heavily skewed (most region pairs share no streamlines at all), while the functional matrix is dense — correlation almost never comes out to exactly zero, even between regions with no direct anatomical connection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))

im0 = axes[0].imshow(np.log1p(structural), cmap="viridis")
axes[0].set_title("Structural (log streamline count)\nStanford HARDI subject", fontsize=12)
fig.colorbar(im0, ax=axes[0], shrink=0.75)

im1 = axes[1].imshow(functional, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title(f"Functional (correlation)\n{functional_subject} — a different person", fontsize=12)
fig.colorbar(im1, ax=axes[1], shrink=0.75)

for ax in axes:
    ax.set_xlabel("region index", fontsize=11)
    ax.set_ylabel("region index", fontsize=11)

plt.tight_layout()
plt.show()

## Edge-wise comparison

Take the upper triangle of both matrices (avoiding the diagonal and each pair's duplicate), so we have one structural value and one functional value per unique region pair. Structural streamline counts are heavily skewed, so we log-transform before comparing — standard practice for this kind of count data.

In [ ]:
iu = np.triu_indices_from(structural, k=1)
struct_edges = structural[iu]
func_edges = functional[iu]
log_struct_edges = np.log1p(struct_edges)

r, p = stats.pearsonr(log_struct_edges, func_edges)
print(f"{len(struct_edges)} region pairs")
print(f"structurally-connected pairs (nonzero streamlines): {(struct_edges > 0).sum()} "
      f"({100 * (struct_edges > 0).mean():.1f}%)")
print(f"\nPearson correlation, log(1+streamlines) vs functional correlation: r={r:.3f}, p={p:.2e}")

## Plotting it

Every point is one region pair. If structure fully determined function, these points would fall on a tight line. They don't.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6.5))
ax.scatter(log_struct_edges, func_edges, s=18, alpha=0.35, color="#2b6cb0")

# a simple linear fit, just to show the trend
coef = np.polyfit(log_struct_edges, func_edges, 1)
xs = np.linspace(log_struct_edges.min(), log_struct_edges.max(), 50)
ax.plot(xs, np.polyval(coef, xs), color="#c53030", linewidth=2.5,
        label=f"linear trend (r={r:.2f})")

ax.set_xlabel("log(1 + structural streamline count)", fontsize=12)
ax.set_ylabel("functional correlation", fontsize=12)
ax.set_title("Structural vs functional connectivity, edge by edge", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Reading this honestly

The correlation is real and clearly not zero (p is tiny — with over 2000 region pairs, even a weak relationship reaches significance easily, which is exactly why the *size* of r matters far more than its p-value here). But r sits around 0.1, which is weak. Published same-subject, same-atlas studies with proper diffusion and resting-state data from the same people typically report structure-function correlations in the range of roughly 0.3 to 0.5 — still far from 1, but noticeably stronger than what we found here. Some of that gap is genuine neuroscience (function is not fully determined by structure, full stop). Some of it is very likely an artefact of exactly the two honesty problems we flagged at the top: different people, and two atlases matched only approximately by name.

One more thing worth checking: does streamline count actually track functional strength *among only the pairs that are structurally connected at all*, or is the correlation above driven mostly by the presence/absence split (connected pairs tend to have higher functional correlation than disconnected ones, regardless of exactly how many streamlines)?

In [ ]:
connected = struct_edges > 0
r_connected, p_connected = stats.pearsonr(log_struct_edges[connected], func_edges[connected])
print(f"structurally-connected pairs only (n={connected.sum()}):")
print(f"  correlation between streamline count and functional strength: r={r_connected:.3f}, p={p_connected:.3f}")
print(f"\nmean functional correlation, structurally connected pairs:    {func_edges[connected].mean():+.3f}")
print(f"mean functional correlation, structurally unconnected pairs:  {func_edges[~connected].mean():+.3f}")

Once you restrict to pairs that already have *some* direct anatomical connection, the exact streamline count stops predicting functional strength at all — the correlation collapses to essentially zero. What's actually driving the overall relationship is closer to a simpler fact: connected pairs tend to run somewhat higher in functional correlation than unconnected pairs, on average, not "more fibres, proportionally tighter coupling". That's a real, replicated pattern in the structure-function literature, and a good example of why edge-wise correlation alone can be misleading if you don't also look at what's driving it.

**Next session:** we leave synthetic and single-subject demonstrations behind for the capstone. Fifty real participants, the same functional dataset from the first course's finale, and every method from sessions 1 through 6 applied properly.